# Recommendation Systems Introduction

This notebook provides a hands-on exploration of recommendation systems using the MovieLens 100k dataset. It begins with loading and examining the dataset, followed by interactive sections to better understand its structure. The core focus is on two major classes of recommendation approaches: memory-based methods, which rely on user–user and item–item similarities (e.g., collaborative filtering with cosine similarity), and model-based methods, which use machine learning techniques to capture latent patterns in the data. By walking through both perspectives, the notebook illustrates the intuition, implementation, and comparative strengths of these approaches in generating personalized movie recommendations.

Please note that you will have to select the Tensorflow kernel in order for this notebook to run properly.

## Imports

In [1]:
import numpy as np
import pandas as pd
import os
import zipfile
import urllib.request
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import NMF

## Dataset

### Downloading Dataset

In [2]:
dataset_url = "http://files.grouplens.org/datasets/movielens/ml-100k.zip"
dataset_path = "ml-100k.zip"
extract_folder = "ml-100k"

if not os.path.exists(extract_folder):
    if not os.path.exists(dataset_path):
        print("Downloading dataset...")
        urllib.request.urlretrieve(dataset_url, dataset_path)
    with zipfile.ZipFile(dataset_path, "r") as zip_ref:
        zip_ref.extractall(".")

### Dataset Setup

In [3]:
ratings = pd.read_csv(
    os.path.join(extract_folder, "u.data"),
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
    encoding="latin-1"
)
print("Ratings shape:", ratings.shape)

Ratings shape: (100000, 4)


In [4]:
ratings.head(3)

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116


In [5]:
movies = pd.read_csv(
    os.path.join(extract_folder, "u.item"),
    sep="|",
    names=["movie_id", "title", "release_date", "video_release_date", "IMDb_URL", "unknown",
           "Action","Adventure","Animation","Children's","Comedy","Crime","Documentary","Drama",
           "Fantasy","Film-Noir","Horror","Musical","Mystery","Romance","Sci-Fi","Thriller",
           "War","Western"],
    encoding="latin-1"
)
print("Movies shape:", movies.shape)

Movies shape: (1682, 24)


In [6]:
movies.head()

,movie_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [7]:
user_item_matrix = ratings.pivot(index="user_id", columns="item_id", values="rating").fillna(0)

In [8]:
user_item_matrix.head(3)

item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Dataset Exploration

In [9]:
movies.head()

,movie_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [10]:
ratings.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [11]:
user_item_matrix.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
# Example from the ratings dataset
example_rating = ratings.iloc[0]
print(f"User {int(example_rating['user_id'])} rated movie {int(example_rating['item_id'])} with a rating of {int(example_rating['rating'])}.")

# Example from the movies dataset
example_movie = movies.iloc[0]
print(f"\nThe movie with ID {int(example_movie['movie_id'])} is titled '{example_movie['title']}' and was released on {example_movie['release_date']}.")

# Example from the user_item_matrix dataset
user = 1
movie = 4
print(f"\nUser {user} rated movie {movie} with a rating of {int(user_item_matrix[movie][user])}.")

User 196 rated movie 242 with a rating of 3.

The movie with ID 1 is titled 'Toy Story (1995)' and was released on 01-Jan-1995.

User 1 rated movie 4 with a rating of 3.


## Memory Based

Memory-based recommendation systems, often referred to as neighborhood methods, work directly with the user–item rating matrix to find similarities between users or items. By leveraging similarity measures such as cosine similarity, these methods identify users with comparable preferences or items with similar rating patterns, and then use this information to generate recommendations. They are conceptually simple, intuitive to implement, and provide a clear explanation of why a particular recommendation was made.

### User Cosine Similarity

In [13]:
user_similarity = cosine_similarity(user_item_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("User-User Similarity Matrix Shape:", user_similarity_df.shape)

User-User Similarity Matrix Shape: (943, 943)


In [14]:
user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.166931,0.047460,0.064358,0.378475,0.430239,0.440367,0.319072,0.078138,0.376544,...,0.369527,0.119482,0.274876,0.189705,0.197326,0.118095,0.314072,0.148617,0.179508,0.398175
2,0.166931,1.000000,0.110591,0.178121,0.072979,0.245843,0.107328,0.103344,0.161048,0.159862,...,0.156986,0.307942,0.358789,0.424046,0.319889,0.228583,0.226790,0.161485,0.172268,0.105798
3,0.047460,0.110591,1.000000,0.344151,0.021245,0.072415,0.066137,0.083060,0.061040,0.065151,...,0.031875,0.042753,0.163829,0.069038,0.124245,0.026271,0.161890,0.101243,0.133416,0.026556
4,0.064358,0.178121,0.344151,1.000000,0.031804,0.068044,0.091230,0.188060,0.101284,0.060859,...,0.052107,0.036784,0.133115,0.193471,0.146058,0.030138,0.196858,0.152041,0.170086,0.058752
5,0.378475,0.072979,0.021245,0.031804,1.000000,0.237286,0.373600,0.248930,0.056847,0.201427,...,0.338794,0.080580,0.094924,0.079779,0.148607,0.071459,0.239955,0.139595,0.152497,0.313941


In [15]:
def recommend_movies(user_id, user_item_matrix, user_similarity, movies, top_n=5):

    # Extract user similarity score with every other user.
    sim_scores = user_similarity[user_id - 1]
    sim_scores = sim_scores.reshape(1, -1)

    # Rescale the ratings giving more weight on similar users
    weighted_ratings = sim_scores.dot(user_item_matrix.values)

    # Produce predictions as an average of the movie rating across users
    sim_sums = np.abs(sim_scores).sum(axis=1)
    pred_ratings = weighted_ratings / (sim_sums + 1e-8)
    preds = pd.Series(pred_ratings.flatten(), index=user_item_matrix.columns)

    # Filter out movies already rated by the user so we do not suggest them again
    watched = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index
    preds = preds.drop(watched)

    # Extract and return the top rated movies for suggestions
    top_movies = preds.sort_values(ascending=False).head(top_n)
    return movies[movies["movie_id"].isin(top_movies.index)][["movie_id", "title"]]

In [66]:
recommend_movies(81, user_item_matrix, user_similarity, movies)

,movie_id,title
49,50,Star Wars (1977)
116,117,"Rock, The (1996)"
126,127,"Godfather, The (1972)"
173,174,Raiders of the Lost Ark (1981)
180,181,Return of the Jedi (1983)


In [65]:
# check current preferences of user
def user_current_preference(user_id, movies, user_item_matrix):
  # keep only titles
  movies_names = movies['title']
  # call all movies from user_item_matrix for user 140 and rank top rating to bottom
  user_movies = user_item_matrix.loc[user_id].sort_values(ascending=False).to_frame()
  user_movies
  # merge movie names on index
  df = user_movies.merge(right = movies_names, how = 'left', left_index = True, right_index = True)
  return df[:10]

user_current_preference(user_id=81, movies=movies, user_item_matrix=user_item_matrix)


,81,title
item_id,,
591,5.0,True Crime (1995)
79,5.0,Hot Shots! Part Deux (1993)
186,5.0,"Godfather: Part II, The (1974)"
98,5.0,Snow White and the Seven Dwarfs (1937)
282,5.0,Emma (1996)
25,5.0,"Brothers McMullen, The (1995)"
475,5.0,"First Wives Club, The (1996)"
318,5.0,Everyone Says I Love You (1996)
1,4.0,GoldenEye (1995)


## Model Based

Model-based recommendation systems go beyond direct similarity calculations and instead build predictive models that capture underlying patterns in the data. A common approach is matrix factorization, where the user–item rating matrix is decomposed into latent factors representing hidden characteristics of users and items. These models are generally more scalable and can handle sparse datasets better than memory-based methods, while often achieving higher accuracy. Although they may be less interpretable, model-based techniques form the foundation of many modern large-scale recommendation systems.

### Matrix Factorization with Singular Value Decomposition (SVD)

In [38]:
# Get the latent factors of users and items using SVD
svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_

# Use them to get a ratings prediction for each pair of a user and a pair
pred_ratings_svd = np.dot(user_factors, item_factors)

In [67]:
def recommend_movies_svd(user_id, pred_ratings, user_item_matrix, movies, top_n=5):
    # Get the predicted rating for the user
    user_pred = pred_ratings[user_id-1]

    # Filter out movies already rated by the user so we do not suggest them again
    watched = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index
    preds = pd.Series(user_pred, index=user_item_matrix.columns).drop(watched)

    # Extract and return the top rated movies for suggestions
    top_movies = preds.sort_values(ascending=False).head(top_n)
    return movies[movies["movie_id"].isin(top_movies.index)][["movie_id", "title"]]

recommend_movies_svd(81, pred_ratings_svd, user_item_matrix, movies)

,movie_id,title
8,9,Dead Man Walking (1995)
14,15,Mr. Holland's Opus (1995)
116,117,"Rock, The (1996)"
124,125,Phenomenon (1996)
545,546,Broken Arrow (1996)


In [69]:
user_current_preference(user_id=81, movies=movies, user_item_matrix=user_item_matrix)


,81,title
item_id,,
591,5.0,True Crime (1995)
79,5.0,Hot Shots! Part Deux (1993)
186,5.0,"Godfather: Part II, The (1974)"
98,5.0,Snow White and the Seven Dwarfs (1937)
282,5.0,Emma (1996)
25,5.0,"Brothers McMullen, The (1995)"
475,5.0,"First Wives Club, The (1996)"
318,5.0,Everyone Says I Love You (1996)
1,4.0,GoldenEye (1995)


### Matrix Factorization with Non-Negative Matrix Factorization (NMF)

In [51]:
# Get the latent factors of users and items using NMF
nmf = NMF(n_components=20, init="random", random_state=42, max_iter=200)
user_factors = nmf.fit_transform(user_item_matrix)
item_factors = nmf.components_

# Use them to get a ratings prediction for each pair of a user and a pair
pred_ratings = np.dot(user_factors, item_factors)

/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [52]:
def recommend_movies_nmf(user_id, pred_ratings, user_item_matrix, movies, top_n=5):
    # Get the predicted rating for the user
    user_pred = pred_ratings[user_id-1]

    # Filter out movies already rated by the user so we do not suggest them again
    watched = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index
    preds = pd.Series(user_pred, index=user_item_matrix.columns).drop(watched)

    # Extract and return the top rated movies for suggestions
    top_movies = preds.sort_values(ascending=False).head(top_n)
    return movies[movies["movie_id"].isin(top_movies.index)][["movie_id", "title"]]

In [70]:
recommend_movies_nmf(81, pred_ratings, user_item_matrix, movies)

,movie_id,title
8,9,Dead Man Walking (1995)
116,117,"Rock, The (1996)"
124,125,Phenomenon (1996)
507,508,"People vs. Larry Flynt, The (1996)"
545,546,Broken Arrow (1996)


### Generalized Matrix Factorization (GMF)

Generalized Matrix Factorization extends the basic idea of matrix factorization by making the interaction function between user and item latent factors more flexible. Instead of relying solely on the dot product (as in traditional MF), GMF introduces a neural network layer to learn how user and item embeddings should be combined to predict ratings or preferences. This allows the model to capture more complex, nonlinear relationships between users and items while still retaining the interpretability and efficiency of embedding-based representations. GMF is a core component in modern neural recommendation architectures such as Neural Collaborative Filtering (NCF).

In [54]:
# Setup the data used to train the model
num_users = ratings["user_id"].nunique()
num_items = ratings["item_id"].nunique()

# Scale the data
ratings["rating_norm"] = ratings["rating"] / 5.0

X = [ratings["user_id"].values - 1, ratings["item_id"].values - 1]
y = ratings["rating_norm"].values
latent_dim = 20

In [55]:
# Input Layers
user_input = keras.Input(shape=(1,), name="user")
item_input = keras.Input(shape=(1,), name="item")

# Embedding Layers
user_emb = layers.Embedding(num_users, latent_dim, name="user_embedding")(user_input)
item_emb = layers.Embedding(num_items, latent_dim, name="item_embedding")(item_input)

# Flatten Embeddings
user_vec = layers.Flatten()(user_emb)
item_vec = layers.Flatten()(item_emb)

# Element-wise multiplication between user and item embeddings
interaction = layers.Multiply()([user_vec, item_vec])
output = layers.Dense(1, activation="sigmoid")(interaction)

# Compile the model
model = keras.Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer="adam", loss="mse")

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 20)     │     18,860 │ user[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 20)     │     33,640 │ item[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 20)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 20)        │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 20)        │          0 │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │         21 │ multiply[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 52,521 (205.16 KB)

 Trainable params: 52,521 (205.16 KB)

 Non-trainable params: 0 (0.00 B)

In [56]:
# Train the model
history = model.fit(
    X, y,
    epochs=5,
    batch_size=512,
    validation_split=0.2,
    verbose=1
)

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0895 - val_loss: 0.0779
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0754 - val_loss: 0.0587
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0510 - val_loss: 0.0413
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0384 - val_loss: 0.0384
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0357 - val_loss: 0.0368


In [57]:
def recommend_gmf_keras(user_id, model, movies, top_n=5):

    # Get the model predictions for the user
    user = np.array([user_id-1] * num_items)
    items = np.arange(num_items)
    preds = model.predict([user, items], verbose=0).flatten()

    # Filter out movies already rated by the user so we do not suggest them again
    watched = ratings[ratings["user_id"] == user_id]["item_id"].values
    preds_filtered = {i+1: preds[i] for i in range(num_items) if (i+1) not in watched}

    # Extract and return the top rated movies for suggestions
    top_items = sorted(preds_filtered.items(), key=lambda x: x[1], reverse=True)[:top_n]
    top_movie_ids = [movie_id for movie_id, _ in top_items]

    return movies[movies["movie_id"].isin(top_movie_ids)][["movie_id", "title"]]

In [71]:
recommend_gmf_keras(81, model, movies)

,movie_id,title
11,12,"Usual Suspects, The (1995)"
49,50,Star Wars (1977)
63,64,"Shawshank Redemption, The (1994)"
126,127,"Godfather, The (1972)"
482,483,Casablanca (1942)


### GMF + MLP

While GMF enriches traditional matrix factorization with learnable interaction functions, combining it with a Multi-Layer Perceptron (MLP) further enhances the model’s ability to capture complex user–item relationships. The GMF component preserves the embedding-based representation and multiplicative interaction, while the MLP introduces nonlinear transformations that can model higher-order correlations. By jointly training both parts and merging their outputs, this hybrid approach—commonly referred to as Neural Collaborative Filtering (NCF)—achieves a balance between interpretability and expressive power, leading to more accurate and flexible recommendations.

In [73]:
latent_dim = 20
mlp_latent_dim = 40

# Input Layers
user_input = layers.Input(shape=(1,), name="user")
item_input = layers.Input(shape=(1,), name="item")

# Embedding Layers
gmf_user_emb = layers.Embedding(num_users, latent_dim, name="gmf_user_embedding")(user_input)
gmf_item_emb = layers.Embedding(num_items, latent_dim, name="gmf_item_embedding")(item_input)

# Flatten Embeddings
gmf_user_vec = layers.Flatten()(gmf_user_emb)
gmf_item_vec = layers.Flatten()(gmf_item_emb)

# Embeddings Combination
gmf_interaction = layers.Multiply()([gmf_user_vec, gmf_item_vec])

# MLP Embedding Layers
mlp_user_emb = layers.Embedding(num_users, mlp_latent_dim, name="mlp_user_embedding")(user_input)
mlp_item_emb = layers.Embedding(num_items, mlp_latent_dim, name="mlp_item_embedding")(item_input)

# MLP Flatten Embeddings
mlp_user_vec = layers.Flatten()(mlp_user_emb)
mlp_item_vec = layers.Flatten()(mlp_item_emb)

# Define MLP Architecture
mlp_layer = layers.Concatenate()([mlp_user_vec, mlp_item_vec])
# In class TASK
mlp_layer = layers.Dense(128, activation='relu')(mlp_layer)
mlp_layer = layers.Dense(64, activation='relu')(mlp_layer)

# Merge GMF and MLP results
merged = layers.Concatenate()([gmf_interaction, mlp_layer])

# Produce Output
output = layers.Dense(1, activation="sigmoid")(merged)

# Compile the model
model = keras.Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer="adam", loss="mse")

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_user_embedding  │ (None, 1, 40)     │     37,720 │ user[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlp_item_embedding  │ (None, 1, 40)     │     67,280 │ item[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_8 (Flatten) │ (None, 40)        │          0 │ mlp_user_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_9 (Flatten) │ (None, 40)        │          0 │ mlp_item_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gmf_user_embedding  │ (None, 1, 20)     │     18,860 │ user[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gmf_item_embedding  │ (None, 1, 20)     │     33,640 │ item[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 80)        │          0 │ flatten_8[0][0],  │
│ (Concatenate)       │                   │            │ flatten_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_6 (Flatten) │ (None, 20)        │          0 │ gmf_user_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_7 (Flatten) │ (None, 20)        │          0 │ gmf_item_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     10,368 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_2          │ (None, 20)        │          0 │ flatten_6[0][0],  │
│ (Multiply)          │                   │            │ flatten_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 84)        │          0 │ multiply_2[0][0], │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         85 │ concatenate_3[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 176,209 (688.32 KB)

 Trainable params: 176,209 (688.32 KB)

 Non-trainable params: 0 (0.00 B)

In [74]:
# Train the model
history = model.fit(
    X, y,
    epochs=5,
    batch_size=512,
    validation_split=0.2,
    verbose=1
)

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - loss: 0.0567 - val_loss: 0.0359
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0345 - val_loss: 0.0349
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0337 - val_loss: 0.0345
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0322 - val_loss: 0.0342
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0299 - val_loss: 0.0337


In [75]:
def recommend_gmf_mlp(user_id, model, movies, top_n=5):

    # Get the model predictions for the user
    user = np.array([user_id-1] * num_items)
    items = np.arange(num_items)
    preds = model.predict([user, items], verbose=0).flatten()


    # Filter out movies already rated by the user so we do not suggest them again
    watched = ratings[ratings["user_id"] == user_id]["item_id"].values
    preds_filtered = {i+1: preds[i] for i in range(num_items) if (i+1) not in watched}

    # Extract and return the top rated movies for suggestions
    top_items = sorted(preds_filtered.items(), key=lambda x: x[1], reverse=True)[:top_n]
    top_movie_ids = [movie_id for movie_id, _ in top_items]

    return movies[movies["movie_id"].isin(top_movie_ids)][["movie_id", "title"]]

In [77]:
recommend_gmf_mlp(81, model, movies)

,movie_id,title
63,64,"Shawshank Redemption, The (1994)"
113,114,Wallace & Gromit: The Best of Aardman Animatio...
407,408,"Close Shave, A (1995)"
602,603,Rear Window (1954)
1448,1449,Pather Panchali (1955)


In [78]:
# show current preferences for user 81
user_current_preference(user_id=81, movies=movies, user_item_matrix=user_item_matrix)

,81,title
item_id,,
591,5.0,True Crime (1995)
79,5.0,Hot Shots! Part Deux (1993)
186,5.0,"Godfather: Part II, The (1974)"
98,5.0,Snow White and the Seven Dwarfs (1937)
282,5.0,Emma (1996)
25,5.0,"Brothers McMullen, The (1995)"
475,5.0,"First Wives Club, The (1996)"
318,5.0,Everyone Says I Love You (1996)
1,4.0,GoldenEye (1995)


### GMF + MLP Splitted Embeddings

An extension of the GMF + MLP framework is to split the user and item embeddings into two parts: one dedicated to the GMF branch and the other to the MLP branch. This design prevents the two components from sharing identical representations and allows each branch to specialize—GMF focuses on learning linear interactions, while MLP learns nonlinear patterns. By combining their complementary strengths at the output layer, this approach enhances the overall expressiveness of the model and often leads to better recommendation accuracy compared to using a single shared embedding space.

In [79]:
# Define inputs
item_input = layers.Input(shape=(1,), name='item-input')
user_input = layers.Input(shape=(1,), name='user-input')

# MLP Embeddings
movie_embedding_mlp = layers.Embedding(num_items + 1, latent_dim, name='movie-embedding-mlp')(item_input)
movie_vec_mlp = layers.Flatten(name='flatten-movie-mlp')(movie_embedding_mlp)

user_embedding_mlp = layers.Embedding(num_users + 1, latent_dim, name='user-embedding-mlp')(user_input)
user_vec_mlp = layers.Flatten(name='flatten-user-mlp')(user_embedding_mlp)

# GMF Embeddings
movie_embedding_mf = layers.Embedding(num_items + 1, latent_dim, name='movie-embedding-mf')(item_input)
movie_vec_mf = layers.Flatten(name='flatten-movie-mf')(movie_embedding_mf)

user_embedding_mf = layers.Embedding(num_users + 1, latent_dim, name='user-embedding-mf')(user_input)
user_vec_mf = layers.Flatten(name='flatten-user-mf')(user_embedding_mf)

# MLP layers
mlp_layer = layers.Concatenate(name='mlp_layer')([movie_vec_mlp, user_vec_mlp])
# Replacing the placeholder with a simple MLP architecture
mlp_layer = layers.Dense(128, activation='relu')(mlp_layer)
mlp_layer = layers.Dense(64, activation='relu')(mlp_layer)


# Prediction from MLP and GMF
pred_mlp = layers.Dense(10, activation='relu', name='pred-mlp')(mlp_layer)

pred_mf = layers.Dot(axes=1, normalize=False, name="pred_mf")([movie_vec_mf, user_vec_mf])

# Concatenate Embeddings
combine_mlp_mf = layers.Concatenate(name='combine-mlp-mf')([pred_mf, pred_mlp])

# Final prediction
result = layers.Dense(1, activation='relu', name='result')(combine_mlp_mf)

# Compile the model
model = keras.Model(inputs=[user_input, item_input], outputs=result)
model.compile(optimizer="adam", loss="mse")

model.summary()

ValueError: Only input tensors may be passed as positional arguments. The following argument value should be passed as a keyword argument: Ellipsis (of type <class 'ellipsis'>)

In [ ]:
# Train the model
history = model.fit(
    X, y,
    epochs=5,
    batch_size=512,
    validation_split=0.2,
    verbose=1
)

In [ ]:
def recommend_gmf_mlp_conc(user_id, model, movies, top_n=5):
    # Get the model predictions for the user
    user = np.array([user_id-1] * num_items)
    items = np.arange(num_items)
    preds = model.predict([user, items], verbose=0).flatten()

    # Filter out movies already rated by the user so we do not suggest them again
    watched = ratings[ratings["user_id"] == user_id]["item_id"].values
    preds_filtered = {i+1: preds[i] for i in range(num_items) if (i+1) not in watched}

    # Extract and return the top rated movies for suggestions
    top_items = sorted(preds_filtered.items(), key=lambda x: x[1], reverse=True)[:top_n]
    top_movie_ids = [movie_id for movie_id, _ in top_items]

    return movies[movies["movie_id"].isin(top_movie_ids)][["movie_id", "title"]]

In [ ]:
recommend_gmf_mlp_conc(10, model, movies)